In [3]:
import pandas as pd
from autogluon.tabular import TabularPredictor
import os
import shutil


def run_training(config):
    # 파일명 생성 로직
    output_filename = f"submission_{config['user_name']}_{config['feature_tag']}.csv"
    model_save_path = f"ag_models_{config['user_name']}_{config['feature_tag']}"

    print(f"{'='*80}")
    print(f"👤 [팀원] {config['user_name']} | 🏷️ [피처 태그] {config['feature_tag']}")
    print(f"📂 학습 데이터: {config['train_file']}")
    print(f"🎯 목표: {config['target_col']} ({config['eval_metric']})")
    print(f"{'='*80}\n")

    # [Tip] 기존에 에러나서 남은 폴더가 있으면 충돌날 수 있으니 삭제 시도
    if os.path.exists(model_save_path):
        try:
            shutil.rmtree(model_save_path)
            print(f"🧹 기존 실패 폴더 삭제 완료: {model_save_path}")
        except:
            print(f"⚠️ 기존 폴더 삭제 실패 (무시하고 진행): {model_save_path}")

    # 1. 데이터 로드
    try:
        train = pd.read_csv(config["train_file"], encoding="utf-8")
        test = pd.read_csv(config["test_file"], encoding="utf-8")
    except UnicodeDecodeError:
        train = pd.read_csv(config["train_file"], encoding="euc-kr")
        test = pd.read_csv(config["test_file"], encoding="euc-kr")

    # ID 제거
    if config["id_col"]:
        train_data = train.drop(columns=[config["id_col"]], errors="ignore")
        test_data = test.drop(columns=[config["id_col"]], errors="ignore")
    else:
        train_data = train
        test_data = test

    # 2. 모델 학습
    print("🤖 AutoGluon 학습 시작...")

    fit_args = {"num_gpus": 1 if config["use_gpu"] else 0}
    kwargs = {}
    if config.get("stack_level"):
        kwargs["num_stack_levels"] = config["stack_level"]

    # ★ [핵심 해결책] 윈도우 에러 방지를 위해 dynamic_stacking을 끕니다.
    kwargs["dynamic_stacking"] = False

    predictor = TabularPredictor(
        label=config["target_col"],
        problem_type="regression",  # ★★★ [필수 추가] 이게 없으면 또 에러 납니다!
        eval_metric=config["eval_metric"],
        path=model_save_path,
        # seed 제거됨
    ).fit(
        train_data,
        presets=config["presets"],
        time_limit=config["time_limit"],
        ag_args_fit=fit_args,
        **kwargs,  # 여기에 dynamic_stacking=False가 들어갑니다.
    )

    # 3. 예측
    print("\n🔮 예측 수행 중...")

    # 회귀 문제는 output_mode가 proba여도 그냥 predict를 써야 합니다.
    if config["output_mode"] == "proba" and predictor.problem_type != "regression":
        if predictor.problem_type == "binary":
            pred = predictor.predict_proba(test_data).iloc[:, 1]
        else:  # multiclass 등
            pred = predictor.predict(test_data)  # 혹은 필요에 따라 로직 변경
    else:
        # 회귀(regression)는 무조건 여기로 와야 함
        pred = predictor.predict(test_data)

    # 4. 저장 (★ 여기가 수정됨: 샘플 파일 없어도 강제 생성)
    try:
        # 일단 샘플 파일 읽기 시도
        try:
            sub = pd.read_csv(config["sample_sub"], encoding="cp949")
        except:
            sub = pd.read_csv(config["sample_sub"], encoding="utf-8")
        print("📄 sample_submission.csv 파일을 사용하여 형식을 맞춥니다.")

    except FileNotFoundError:
        # 샘플 파일이 없으면 테스트 데이터에서 ID 가져와서 새로 만듦
        print("⚠️ sample_submission.csv 없음! 테스트 데이터의 ID로 강제 생성합니다.")
        sub = pd.DataFrame()
        if config["id_col"] in test.columns:
            sub[config["id_col"]] = test[config["id_col"]]
        else:
            # ID 컬럼도 없으면 그냥 인덱스나 첫번째 컬럼 사용
            sub[config["id_col"]] = test.iloc[:, 0]

    # 예측값 채우기
    if isinstance(pred, pd.DataFrame):
        cols = [c for c in sub.columns if c != config["id_col"]]
        sub[cols] = pred.values
    else:
        sub[config["target_col"]] = pred

    sub.to_csv(output_filename, index=False)
    print(f"\n✅ 저장 완료: {output_filename}")
    print(f"   (이 파일을 리더보드에 제출하고 점수를 기록해두세요!)")


# ==============================================================================
# ▼ 설정값 (SEM - GPU 버전)
# ==============================================================================
MY_CONFIG = {
    "user_name": "SEM",
    # ★ 버전을 v3로 올렸습니다 (혹시 모를 폴더 잠금 회피)
    "feature_tag": "Crazy_Unsupervised_v3",
    "train_file": "train_for_autogluon.csv",
    "test_file": "test_for_autogluon.csv",  # ★ 주의: 파일명이 test_C.csv가 아니라 X_test.csv가 맞는지 확인하세요!
    "sample_sub": "sample_submission.csv",
    "target_col": "age",
    "id_col": "custid",
    "presets": "high_quality",
    "eval_metric": "root_mean_squared_error",
    "time_limit": 1200,  # ★ 테스트용으로 100초 해두신 것 같은데, 실제 돌릴 땐 3600으로 늘리세요!
    "random_seed": 777,
    "stack_level": None,
    "output_mode": "proba",
    "use_gpu": True,
}

if __name__ == "__main__":
    run_training(MY_CONFIG)

👤 [팀원] SEM | 🏷️ [피처 태그] Crazy_Unsupervised_v3
📂 학습 데이터: train_for_autogluon.csv
🎯 목표: age (root_mean_squared_error)

⚠️ 기존 폴더 삭제 실패 (무시하고 진행): ag_models_SEM_Crazy_Unsupervised_v3


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.7
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Memory Avail:       5.79 GB / 15.21 GB (38.0%)
Disk Space Avail:   231.98 GB / 475.89 GB (48.7%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.


🤖 AutoGluon 학습 시작...


Beginning AutoGluon training ... Time limit = 1200s
AutoGluon will save models to "c:\SEMIN\Project\AUTOML_PIPELINE\ag_models_SEM_Crazy_Unsupervised_v3"
Train Data Rows:    21587
Train Data Columns: 636
Label Column:       age
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    5721.75 MB
	Train Data (Original)  Memory Usage: 107.51 MB (1.9% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDupl


🔮 예측 수행 중...
⚠️ sample_submission.csv 없음! 테스트 데이터의 ID로 강제 생성합니다.

✅ 저장 완료: submission_SEM_Crazy_Unsupervised_v3.csv
   (이 파일을 리더보드에 제출하고 점수를 기록해두세요!)
